# VideoAnnotator results in Python

This notebook reads VideoAnnotator's **results table**: every pipeline's results in one CSV, one
row per event and measure, the same columns for every pipeline. It walks through a small analysis
of the demo clip (a parent and infant playing peekaboo) that ships beside this notebook.

**To use your own results**, export the table (the viewer's *Export → Table (CSV)* on a run or
job page, or `videoannotator export table <results folder>`) and change `TABLE` below. Everything
else runs unchanged; sections whose pipeline you didn't run say so and move on.

Packages: `pandas`, `matplotlib`. The columns are documented in
[`docs/usage/exporting_results.md`](../../docs/usage/exporting_results.md).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

TABLE = "demo_clip_table.csv"

df = pd.read_csv(TABLE)
print(
    f"{len(df):,} rows, {df['video'].nunique()} video(s), {df['pipeline'].nunique()} pipelines"
)
df.head()

## What's in the table

Each row is one **measure** of one **event**. An event is something a pipeline found: a person
in a frame, a face, a speech segment, a speaker turn, a scene, a VLM label. Its first row says
what it is (`person`, `face`, `speech`…); further rows with the same `event` number are things
measured about it (an emotion, an action-unit intensity, gaze). `start` and `end` are seconds from
the start of the video; `end` is empty for something seen at one moment (a frame).

In [ ]:
df.groupby(["pipeline", "measure"]).size().rename("rows").to_frame()

## Who speaks, and for how long

Speaker turns come from diarization (`speaker_turn` rows; `track` is the speaker). Speaker labels
are anonymous (`SPEAKER_00`, …): diarization tells voices apart, it doesn't know who they are.

In [ ]:
turns = df[df["measure"] == "speaker_turn"].copy()
if turns.empty:
    print("No speaker diarization in this table.")
else:
    turns["duration"] = turns["end"] - turns["start"]
    talk = turns.groupby(["video", "track"])["duration"].agg(["count", "sum"])
    talk.columns = ["turns", "seconds"]
    print(talk)

## What was said

Speech recognition gives one `speech` row per segment, with the text in `label`.

In [ ]:
speech = df[df["measure"] == "speech"][["video", "start", "end", "label"]]
speech if not speech.empty else "No speech recognition in this table."

## Who is in view, when

Person tracking gives one `person` row per detection, with a `track` that follows the same person
across frames and a `label` (`parent`, `infant`, …). The labels come from a size heuristic, so
check them: in the demo clip both detected people are labelled `parent`.

In [ ]:
people = df[df["measure"] == "person"]
if people.empty:
    print("No person tracking in this table.")
else:
    print(people.groupby(["track", "label"])["start"].agg(["count", "min", "max"]))
    fig, ax = plt.subplots(figsize=(8, 2.5))
    for i, (track, rows) in enumerate(people.groupby("track")):
        ax.scatter(rows["start"], [i] * len(rows), marker="|", s=200, label=track)
    ax.set_yticks(range(people["track"].nunique()), sorted(people["track"].unique()))
    ax.set_xlabel("seconds")
    ax.set_title("Detections per tracked person")
    plt.tight_layout()
    plt.show()

## Facial expression over time

Face pipelines measure several things per face. `emotion` rows hold the most likely expression in
`label` with its probability in `confidence`; `emotion_score` rows hold every expression's score
(`label` names it, `value` is the score). OpenFace adds `action_unit` rows (one per action unit).

In [ ]:
emotions = df[df["measure"] == "emotion"]
if emotions.empty:
    print("No facial expression results in this table.")
else:
    print(emotions.groupby(["pipeline", "label"]).size().rename("faces"))

units = df[df["measure"] == "action_unit"]
if not units.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    for unit, rows in units.groupby("label"):
        means = rows.groupby("start")["value"].mean()
        ax.plot(means.index, means.values, marker=".", label=unit)
    ax.set_xlabel("seconds")
    ax.set_ylabel("intensity (mean over faces)")
    ax.set_title("OpenFace action units")
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()

## One row per event: going wide

The long layout keeps every pipeline in the same columns. For per-face analysis you often want one
row per face with a column per action unit: that's one `pivot_table` call.

In [ ]:
if units.empty:
    print("No OpenFace results in this table.")
else:
    wide = units.pivot_table(
        index=["video", "pipeline", "event", "start"], columns="label", values="value"
    ).reset_index()
    print(wide.shape)
    print(wide.head())

## Who is in view while someone speaks

Combining pipelines is a join on time. For each speaker turn, which tracked people were detected
during it? People are sampled at moments, so a person counts as in view during a turn if any of
their detections falls inside it.

In [ ]:
if turns.empty or people.empty:
    print("Needs both speaker diarization and person tracking.")
else:
    rows = []
    for _, turn in turns.iterrows():
        seen = people[
            (people["video"] == turn["video"])
            & (people["start"] >= turn["start"])
            & (people["start"] <= turn["end"])
        ]["track"].unique()
        rows.append(
            {
                "video": turn["video"],
                "speaker": turn["track"],
                "start": turn["start"],
                "end": turn["end"],
                "people_in_view": ", ".join(sorted(seen)) or "(none detected)",
            }
        )
    print(pd.DataFrame(rows).to_string(index=False))

## What made these results

Every row names the models (`model`) and VideoAnnotator version that produced it; `not recorded`
means the file came from a version that didn't record it. For a citable methods paragraph with
exact weight revisions, use *Export → Methods paragraph* or `videoannotator export methods`.

In [ ]:
df.groupby(["pipeline", "model", "videoannotator_version"]).size().rename(
    "rows"
).to_frame()